```{contents}
```
## Batch and Epoch

### Motivation and Intuition

Training a neural network means minimizing a loss function over a dataset.
Since datasets are often too large to process at once, optimization is performed **iteratively** using subsets of data and repeated passes over the dataset.

Two fundamental concepts control this process:

| Term      | Meaning                                                             |
| --------- | ------------------------------------------------------------------- |
| **Batch** | Number of training samples processed in one forward–backward update |
| **Epoch** | One complete pass through the entire training dataset               |

---

### Formal Definitions

**Dataset size**:
$$
N = \text{total number of samples}
$$

**Batch size**:
$$
B = \text{samples per batch}
$$

**Number of iterations per epoch**:
$$
\text{steps per epoch} = \left\lceil \frac{N}{B} \right\rceil
$$

**Epoch**:
After all batches are processed once, one **epoch** is completed.

---

### Optimization Workflow

```
for epoch in range(num_epochs):
    for batch in dataset:
        forward pass
        compute loss
        backward pass
        update parameters
```

Graphically:

```
Epoch
 ├── Batch 1 → update weights
 ├── Batch 2 → update weights
 ├── Batch 3 → update weights
 ...
 └── Batch k → update weights
```

---

### Types of Gradient Descent (By Batch Size)

| Method            | Batch Size | Properties                                |
| ----------------- | ---------- | ----------------------------------------- |
| **Stochastic GD** | 1          | Noisy updates, fast convergence, unstable |
| **Mini-batch GD** | 32–512     | Best trade-off, standard practice         |
| **Full Batch GD** | N          | Stable but slow, high memory cost         |

---

### Why Not One Huge Batch?

* GPU memory limits
* Poor generalization for very large batches
* Fewer weight updates → slower convergence

---

### Practical Example with PyTorch

#### Dataset and Model

```python
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Fake dataset
X = torch.randn(1000, 10)
y = torch.randn(1000, 1)

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

model = nn.Linear(10, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
```

#### Training Loop with Epochs and Batches

```python
num_epochs = 5

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}")
    
    for i, (xb, yb) in enumerate(loader):
        # Forward
        pred = model(xb)
        loss = loss_fn(pred, yb)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if i % 10 == 0:
            print(f" Batch {i}, Loss: {loss.item():.4f}")
```

Interpretation:

* **One epoch** = 1000 samples seen once
* **Batch size** = 64 → ~16 updates per epoch
* **5 epochs** → each sample seen 5 times

---

### Training Dynamics

| Quantity      | Effect                                |
| ------------- | ------------------------------------- |
| Larger batch  | More stable gradient, slower learning |
| Smaller batch | Noisy gradient, faster exploration    |
| More epochs   | Better fitting, risk of overfitting   |
| Fewer epochs  | Underfitting                          |

---

### Typical Hyperparameter Ranges

| Parameter  | Common Values                          |
| ---------- | -------------------------------------- |
| Batch size | 32, 64, 128, 256                       |
| Epochs     | 10–100+ (depends on task & model size) |

---

### Advanced Notes

**Effective learning rate coupling**

$$
\text{effective LR} \approx \text{LR} \times \text{Batch Size}
$$

Large batches often require increasing learning rate proportionally.

**Epoch ≠ convergence**
An epoch is a bookkeeping concept; convergence depends on loss behavior, not epoch count.

---

### Summary

| Concept  | Role                                                    |
| -------- | ------------------------------------------------------- |
| Batch    | Controls memory usage, gradient noise, update frequency |
| Epoch    | Controls how many times the model sees the full dataset |
| Training | Nested loops: epochs → batches → weight updates         |

These two parameters fundamentally shape the **speed**, **stability**, and **generalization** of neural network training.